# OME_ZARR explorer

Zarr files are pyramid structures that allow easy access to large imaging datasets. High Content, high resolution images can reach the 100TBs easily. One of the main problems of imaging data, apart from storage is that processing and displaying images at high resolution is resource intensive. This is to the most level solved by using the file Zarr structure, leading to OME-Zarrs and what we call OME-NGFF.

If wou wanna know more about this file structure and how images are stored, you can go here:
- Zarr website
- Fractal
- OME-NGFF

I have also further explained in the README file in section

-----

### Packages that allow access to OME-Zarr files

One requires more preparation than for opening tiff files or png, since the images are stored in binary files hidden at the bottom of the pyramid. 

There are a few packages that facilitate this:
- NGIO: 
- EZ-Zarr
- Napari

In this course we will look at all of this, so that you feel comfortable with opening zarr formatted imaging data.

There are more information on the README file

---------

## EZ-Zarr


One has to first setup the imports neccesary for running the code. Here below you can see the neccesary for dealing with ez-zarr operations.

In [2]:
# Loading the neccesary packages
from config import * #import variables from a hidden file, so in your case you can ignore this - you can set paths explicitly 


import numpy # allows simple operations on strings, arrays, etc.
from pathlib import Path #deals with correct formatting of paths, and path operations liek adding new sections to a path


from ez_zarr import ome_zarr, plotting, utils # one of the packages that allows simple interactions with zarr files


import matplotlib.pyplot as plt # for more fine tuned and complex plotting

Here we will next set the paths to the data

In [3]:
# Paths to change 
zarr_parent_path = zarr_parent_path # change this to add your own path to the script
image_path_prefix = "B/03/0"

# Setting paths
zarr_path = Path(zarr_parent_path,"Day8_Normal_10x_Maximum.zarr") # Setup the path to the data
image_path = Path(zarr_path,image_path_prefix) # Add the extra for well location

#### Loading your first image from ome-zarr
Below you have the code to load a single well, then we will shows how to check information of this well and how to plot the image.

In [4]:
# Loading a single well image

imageA = ome_zarr.Image(image_path)

### Loading your first plate
Plates as you know are made of wells, an ome-zarr normally represents a single plate, and within this ome-zarr we thus have all wells. This can be accessed independtly but one can also access entire plates. Note: This will take longer, loading may take a few minutes.

In [5]:
# Loading a plate
plateL = ome_zarr.import_plate(str(zarr_path)) # returns the plate list - list of images, input has to be a string instead of a Path type

#### Dealing with plates
The next step when you have a plate, is to check the layout and extract a well of interest.

##### 1. Plotting

In [ ]:
plateL.plot()

##### 2. Layout

In [ ]:
plateL.get_layout()

##### 3. Extract one well from plate

This can be done by index in the list or explicitly by well name

In [22]:
well_single = plateL[0]
# or
well_B02 = plateL['B02']

## Dealing with wells
We have learned how to directly load a well and as well how to extract it from a plate. Now we can explore all the possibilities that one can explore from a single well

##### 1. What is contained in a single well object

In [23]:
well_B02

Image B02
  path: /cluster/work/liberali/zarr-data/maaraujo/fractal/20260515_IBD_stain_test/Day8_Normal_10x_Maximum.zarr/B/02/0
  n_channels: 3 (Sytox Green, 568, 647)
  n_pyramid_levels: 5
  pyramid_zyx_scalefactor: [1. 2. 2.]
  full_resolution_zyx_spacing (micrometer): [1.0, 0.6485731390939933, 0.6485731390939933]
  segmentations: Sytox Green_segmented
  tables (measurements): FOV_ROI_table, region_props_features, well_ROI_table, Sytox Green_segmented_masking_ROI_table

In [24]:
imageA

Image 0
  path: /cluster/work/liberali/zarr-data/maaraujo/fractal/20260515_IBD_stain_test/Day8_Normal_10x_Maximum.zarr/B/03/0
  n_channels: 3 (Sytox Green, 568, 647)
  n_pyramid_levels: 5
  pyramid_zyx_scalefactor: [1. 2. 2.]
  full_resolution_zyx_spacing (micrometer): [1.0, 0.6485731390939933, 0.6485731390939933]
  segmentations: Sytox Green_segmented
  tables (measurements): Sytox Green_segmented_masking_ROI_table, FOV_ROI_table, region_props_features, well_ROI_table

You will notice that depending on if it was accessed from the plate or directly through a path loading, some plate localization information may be displayed differently.

##### 2. Plotting 

For single wells we have two ways to do this through matplotlib pyplot or through the use of the ez-zarr package.

In [ ]:
arr = imageA.get_array_by_coordinate()
with plt.style.context('dark_background'):

    fig = plt.figure(figsize=(4, 4)) 

    fig.set_dpi(300)

    plt.imshow(arr[2,0], cmap='gray', vmin=100, vmax=600) # arr[2,0] establishes the channel used for plotting, currently channel 2
    plt.title(imageA.name)
    plt.show()
    plt.close()


In [ ]:
imageA.plot(pyramid_level=0,channels=[1],
         channel_colors=['white'],
         channel_ranges=[[100, 1000]],
         title=imageA.name,
         scalebar_micrometer=150,
         scalebar_color='yellow',
         scalebar_position='topleft',
         scalebar_label=True,
         fig_width_inch=15,
         fig_height_inch=10,
         fig_dpi=600)

##### 3. Segmentations and Tables

In [29]:
imageA

Image 0
  path: /cluster/work/liberali/zarr-data/maaraujo/fractal/20260515_IBD_stain_test/Day8_Normal_10x_Maximum.zarr/B/03/0
  n_channels: 3 (Sytox Green, 568, 647)
  n_pyramid_levels: 5
  pyramid_zyx_scalefactor: [1. 2. 2.]
  full_resolution_zyx_spacing (micrometer): [1.0, 0.6485731390939933, 0.6485731390939933]
  segmentations: Sytox Green_segmented
  tables (measurements): Sytox Green_segmented_masking_ROI_table, FOV_ROI_table, region_props_features, well_ROI_table

In [37]:
df = imageA.get_table(table_name='Sytox Green_segmented_masking_ROI_table')
df

,x_micrometer,y_micrometer,z_micrometer,len_x_micrometer,len_y_micrometer,len_z_micrometer
label,,,,,,
3,1944.422271,1293.903412,0.0,175.763321,70.045899,1.0
1,3570.395131,518.209938,0.0,227.000599,258.132109,1.0
2,3136.499701,1070.145680,0.0,389.143883,419.626821,1.0
4,1037.068449,1315.954899,0.0,339.203752,198.463381,1.0
5,1711.584514,1747.256037,0.0,356.715227,356.066653,1.0
7,2489.872281,2349.131910,0.0,379.415286,396.278188,1.0
8,1470.315306,2526.192377,0.0,404.061066,334.015167,1.0
6,4217.022550,1912.642187,0.0,94.691678,112.851726,1.0
9,1604.569946,2906.256236,0.0,354.120934,400.169627,1.0


This package is very user friendly but has some limitations which can see above, when trying to load a feature table. These table types are not in the correct format, and thus cannot be dealt with this package. This is one of the reasons to also work using the next package NGIO.

In [ ]:
# Add segmentation masks to plotting, try plot only the area of one masks

## NGIO: 

This packages is 

In [ ]:
from pathlib import Path

from ngio import open_ome_zarr_plate, open_ome_zarr_well,open_ome_zarr_container
from ngio_helpers import ZarrWellIterator, Formatter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ngio import open_ome_zarr_well, open_image

In [ ]:
plates = {}
plates["Day8_IFN_10x_Maximum.zarr"] = Path("/cluster/work/liberali/zarr-data/maaraujo/fractal/20260515_IBD_stain_test/Day8_IFN_10x_Maximum.zarr")
plates["Day8_Normal_10x_Maximum.zarr"] = Path("/cluster/work/liberali/zarr-data/maaraujo/fractal/20260515_IBD_stain_test/Day8_Normal_10x_Maximum.zarr")

In [ ]:

well_path =  Path(plates["Day8_Normal_10x_Maximum.zarr"]/"E/02")   # <- your well
level      = "0"                       # "0" = full resolution
out_dir    = Path("./output_for_poster")
use_window = True                      # False -> percentile stretch

row, col  = well_path.parent.name, well_path.name
well_name = f"{well_path.parent.parent.stem}_{row}{col}"

# --- load the whole well -----------------------------------------------------
well  = open_ome_zarr_well(well_path, mode="r")
image = open_image(well_path / well.paths()[0], path=level, mode="r")
print(image.dimensions, image.meta.paths, image.channel_labels)

axes = [a for a in ("t", "c", "z", "y", "x") if image.has_axis(a)]
arr  = image.get_array(axes_order=axes, mode="dask")

if "t" in axes:                                   # first timepoint
    arr = arr[0]; axes.remove("t")
if "z" in axes:                                   # max projection
    arr = arr.max(axis=axes.index("z")); axes.remove("z")
if "c" not in axes:
    arr = arr[None]
planes = arr.compute().astype(np.float32)         # -> (c, y, x)

# --- channel metadata --------------------------------------------------------
channels = image.channels_meta.channels

def hex_to_rgb(h):
    h = h.lstrip("#")
    return np.array([int(h[i:i + 2], 16) for i in (0, 2, 4)]) / 255

def normalise(plane, ch):
    vis = ch.channel_visualisation
    if use_window and vis.end > vis.start:
        lo, hi = vis.start, vis.end
    else:
        lo, hi = np.percentile(plane[::8, ::8], [1, 99.7])
    return np.clip((plane - lo) / max(hi - lo, 1e-6), 0, 1)

labels = [ch.label for ch in channels]
colors = [hex_to_rgb(ch.channel_visualisation.color) for ch in channels]
norm   = [normalise(p, ch) for p, ch in zip(planes, channels)]

views  = [n[..., None] * c for n, c in zip(norm, colors)]
merged = np.clip(sum(views), 0, 1)
views += [merged]
titles = labels + ["merged"]

# --- figure ------------------------------------------------------------------
h, w, panel = planes.shape[1], planes.shape[2], 5.0
fig, axs = plt.subplots(1, len(views), facecolor="#0E1116",
                        figsize=(panel * len(views), panel * h / w + 0.8))

for ax, img, title in zip(np.atleast_1d(axs), views, titles):
    ax.imshow(img, interpolation="nearest")
    ax.set_title(title, color="#C9D1D9", fontsize=11, pad=6)
    ax.axis("off")

fig.suptitle(f"well {row}{col}", color="#E6EDF3", fontsize=14, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(out_dir / f"{well_name}.png", dpi=300,
            facecolor="#0E1116", bbox_inches="tight")


In [ ]:
# full-pixel-resolution versions of each panel
for img, title in zip(views, titles):
    plt.imsave(out_dir / f"{well_name}_{title}.png", img)